# Process Model Improvement

In [ ]:
# Filter data: only start events
import pandas as pd

from preprocessing import load_raw_log

pd.set_option('display.max_colwidth', None)

raw = load_raw_log()

# A_ and O_ events only have a 'complete' event, so they are kept. W_ events are reduced to their 'start' event
df_start = raw[(raw['EventOrigin'] != 'Workflow') | (raw['lifecycle:transition'] == 'start')]

print(f"Original log: {len(raw):,} events")
print(f"Start events: {len(df_start):,} events, {df_start['case:concept:name'].nunique():,} cases")

In [ ]:
# Filter data: Group by variants and focus on the most common variants:
case_variants = df_start.groupby('case:concept:name')['concept:name'].agg(' > '.join)

variants = case_variants.value_counts().rename_axis('variant').reset_index(name='cases')
variants['share'] = variants['cases'] / variants['cases'].sum()
variants['cumulative share'] = variants['share'].cumsum()

print(f"{len(variants):,} variants")
variants.head(60)

In [ ]:
# Discard the variants with a low share
MIN_SHARE = 0.003

kept_variants = variants[variants['share'] >= MIN_SHARE]
kept_cases = case_variants[case_variants.isin(kept_variants['variant'])].index
df_kept = df_start[df_start['case:concept:name'].isin(kept_cases)]

print(f"Kept {len(kept_variants)} of {len(variants):,} variants")
print(f"Kept {len(kept_cases):,} of {len(case_variants):,} cases ({len(kept_cases) / len(case_variants):.1%})")

In [ ]:
# Filter activities: How often did they occur?
activity_counts = pd.DataFrame({
    'events': df_kept['concept:name'].value_counts(),
    'cases': df_kept.groupby('concept:name')['case:concept:name'].nunique(),
})
activity_counts['share of cases'] = (activity_counts['cases'] / len(kept_cases)).round(3)
activity_counts.sort_values('cases', ascending=False)

In [ ]:
# Inspect variants after an activity: which activities follow it, and in which order?
def variants_after(log, activity, steps=3):
    # Every occurrence of the activity counts, steps=None shows the rest of the case
    sequences = []
    for trace in log.groupby('case:concept:name')['concept:name'].agg(list):
        for i, name in enumerate(trace):
            if name == activity:
                following = trace[i + 1:] if steps is None else trace[i + 1:i + 1 + steps]
                sequences.append(' > '.join(following) or '(end of case)')
    table = pd.Series(sequences).value_counts().rename_axis(f'after {activity}').reset_index(name='occurrences')
    table['share'] = (table['occurrences'] / len(sequences)).round(3)
    return table

# manually change this and explore the process
variants_after(df_kept, 'O_Sent (mail and online)', steps=3)

In [ ]:
# Process model: Inductive Miner on the filtered data set
import pm4py

net, im, fm = pm4py.discover_petri_net_inductive(df_kept)
pm4py.view_petri_net(net, im, fm)

## Resulting process model after filtering

In [ ]:
# Save the model as BPMN and evaluate the saved BPMN
from evaluation import evaluate_model

BPMN_PATH = '../output/inductive_miner_filtered.bpmn'

bpmn = pm4py.convert_to_bpmn(net, im, fm)  # pyright: ignore[reportArgumentType]  (pm4py's type hint is wrong)
pm4py.write_bpmn(bpmn, BPMN_PATH)
print(f'Saved {BPMN_PATH}')

bpmn_model = pm4py.convert_to_petri_net(pm4py.read_bpmn(BPMN_PATH))
pd.DataFrame({
    'filtered cases (df_kept)': evaluate_model(df_kept, bpmn_model),
    'all cases (df_start)': evaluate_model(df_start, bpmn_model),
}).round(3)

## Manually adjusted process model

The filtered model above was refined manually in bpmn.io (rare activities, unused gateways and loops removed) and saved as `process_model/final_model.bpmn`. Its evaluation on all cases (`df_start`) is the column *Final* of Table 2 in the paper.

In [ ]:
BPMN_PATH = '../process_model/final_model.bpmn'

bpmn_model = pm4py.convert_to_petri_net(pm4py.read_bpmn(BPMN_PATH))
pd.DataFrame({
    'filtered cases (df_kept)': evaluate_model(df_kept, bpmn_model),
    'all cases (df_start)': evaluate_model(df_start, bpmn_model),
}).round(3)